# Project goal
Revenue is increasing, but profitability is deteriorating.

## Main business question
What should leadership do next to stop margin erosion and improve profitability?

## North star metric
Gross Margin %

In [5]:
# =========================================================
# OPERATIONS PROFITABILITY & COST OPTIMIZATION ANALYSIS
# FINAL FULL DATASET GENERATION SCRIPT
# One-cell Colab script:
# - Mount Google Drive
# - Create project folders
# - Define assumptions
# - Create data dictionary
# - Create locations, products, vendor_prices, sales, costs, waste
# - Export CSV files
# =========================================================

# -----------------------------
# 1. MOUNT GOOGLE DRIVE
# -----------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# -----------------------------
# 2. IMPORTS
# -----------------------------
from pathlib import Path
import pandas as pd
import numpy as np
import calendar

np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# -----------------------------
# 3. PROJECT PATHS
# -----------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/operations-profitability-cost-optimization")
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
SQL_DIR = PROJECT_DIR / "sql"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
DASHBOARD_DIR = PROJECT_DIR / "dashboard"
REPORTS_DIR = PROJECT_DIR / "reports"
ASSETS_DIR = PROJECT_DIR / "assets"

for folder in [
    PROJECT_DIR,
    DATA_DIR,
    RAW_DIR,
    CLEANED_DIR,
    SQL_DIR,
    NOTEBOOKS_DIR,
    DASHBOARD_DIR,
    REPORTS_DIR,
    ASSETS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("PROJECT FOLDERS READY")
print("=" * 100)
print(PROJECT_DIR)

# -----------------------------
# 4. BUSINESS ASSUMPTIONS
# -----------------------------
synthetic_company = "Harvest Table Group"
north_star_metric = "Gross Margin %"
business_problem = "Revenue is increasing, but profitability is deteriorating."
decision_question = "What should leadership do next to stop margin erosion and improve profitability?"

# Time range: Jan 2024 -> Apr 2026
START_MONTH = "2024-01"
END_MONTH = "2026-04"
months = pd.period_range(START_MONTH, END_MONTH, freq="M").astype(str).tolist()

problem_categories = ["Proteins", "Produce"]
high_volume_low_margin_categories = ["Beverages", "Sides"]
underperforming_locations = ["LOC006", "LOC008"]   # Hamilton, London
inflation_vendor_id = "V003"

# Store base performance multipliers
location_sales_factor = {
    "LOC001": 1.20,  # Toronto Downtown
    "LOC002": 1.10,  # North York
    "LOC003": 1.00,  # Scarborough
    "LOC004": 1.08,  # Mississauga
    "LOC005": 0.98,  # Vaughan
    "LOC006": 0.88,  # Hamilton -> underperformer
    "LOC007": 1.05,  # Ottawa
    "LOC008": 0.84,  # London -> underperformer
}

location_waste_factor = {
    "LOC001": 1.00,
    "LOC002": 1.02,
    "LOC003": 1.00,
    "LOC004": 1.03,
    "LOC005": 1.00,
    "LOC006": 1.18,
    "LOC007": 1.00,
    "LOC008": 1.24,
}

location_labor_factor = {
    "LOC001": 1.08,
    "LOC002": 1.04,
    "LOC003": 1.00,
    "LOC004": 1.03,
    "LOC005": 1.00,
    "LOC006": 1.05,
    "LOC007": 1.02,
    "LOC008": 1.01,
}

# Revenue grows, but cost pressure grows faster in later periods
monthly_sales_growth = {
    "2024-01": 1.00, "2024-02": 1.01, "2024-03": 1.02, "2024-04": 1.03, "2024-05": 1.05, "2024-06": 1.06,
    "2024-07": 1.07, "2024-08": 1.08, "2024-09": 1.09, "2024-10": 1.10, "2024-11": 1.11, "2024-12": 1.13,
    "2025-01": 1.14, "2025-02": 1.16, "2025-03": 1.18, "2025-04": 1.20, "2025-05": 1.22, "2025-06": 1.24,
    "2025-07": 1.26, "2025-08": 1.28, "2025-09": 1.29, "2025-10": 1.31, "2025-11": 1.33, "2025-12": 1.35,
    "2026-01": 1.36, "2026-02": 1.38, "2026-03": 1.39, "2026-04": 1.40,
}

monthly_cogs_growth = {
    "2024-01": 1.00, "2024-02": 1.01, "2024-03": 1.03, "2024-04": 1.05, "2024-05": 1.07, "2024-06": 1.09,
    "2024-07": 1.11, "2024-08": 1.13, "2024-09": 1.15, "2024-10": 1.17, "2024-11": 1.19, "2024-12": 1.21,
    "2025-01": 1.23, "2025-02": 1.26, "2025-03": 1.29, "2025-04": 1.33, "2025-05": 1.37, "2025-06": 1.40,
    "2025-07": 1.44, "2025-08": 1.48, "2025-09": 1.51, "2025-10": 1.55, "2025-11": 1.59, "2025-12": 1.63,
    "2026-01": 1.67, "2026-02": 1.71, "2026-03": 1.75, "2026-04": 1.79,
}

waste_multiplier_by_category = {
    "Proteins": 1.55,
    "Produce": 1.40,
    "Beverages": 0.65,
    "Sides": 0.88,
    "Desserts": 0.80,
    "Disposables": 0.55,
}

# Days of week behavior
dow_factor = {
    0: 0.95,   # Monday
    1: 0.97,   # Tuesday
    2: 1.00,   # Wednesday
    3: 1.03,   # Thursday
    4: 1.10,   # Friday
    5: 1.14,   # Saturday
    6: 1.06,   # Sunday
}

# Product-level daily demand seeds
base_units_by_category = {
    "Proteins": 18,
    "Produce": 20,
    "Beverages": 32,
    "Sides": 28,
    "Desserts": 14,
    "Disposables": 36,
}

discount_rate_by_category = {
    "Proteins": 0.04,
    "Produce": 0.03,
    "Beverages": 0.02,
    "Sides": 0.03,
    "Desserts": 0.04,
    "Disposables": 0.00,
}

avg_ticket_base_by_location = {
    "LOC001": 19.50,
    "LOC002": 18.20,
    "LOC003": 17.40,
    "LOC004": 18.60,
    "LOC005": 17.80,
    "LOC006": 16.40,
    "LOC007": 18.10,
    "LOC008": 15.90,
}

waste_reason_weights = {
    "Spoilage": 0.32,
    "Overproduction": 0.28,
    "Expired Inventory": 0.18,
    "Prep Loss": 0.15,
    "Damaged Handling": 0.07,
}

recorded_by_options = ["Kitchen Lead", "Operations Supervisor", "Shift Manager", "Inventory Coordinator"]

vendors = [
    {"vendor_id": "V001", "vendor_name": "Maple Fresh Foods"},
    {"vendor_id": "V002", "vendor_name": "Northern Supply Co"},
    {"vendor_id": "V003", "vendor_name": "Prime Protein Partners"},
    {"vendor_id": "V004", "vendor_name": "Green Harvest Produce"},
    {"vendor_id": "V005", "vendor_name": "TrueServe Packaging"},
]

print("\n" + "=" * 100)
print("BUSINESS ASSUMPTIONS")
print("=" * 100)
print(f"Synthetic company: {synthetic_company}")
print(f"North star metric: {north_star_metric}")
print(f"Time period: {months[0]} to {months[-1]}")
print(f"Problem categories: {problem_categories}")
print(f"Underperforming locations: {underperforming_locations}")
print(f"Inflation vendor: {inflation_vendor_id}")

# -----------------------------
# 5. CREATE DIMENSION TABLES
# -----------------------------
locations_data = [
    {"location_id": "LOC001", "location_name": "Toronto Downtown", "city": "Toronto",     "region": "Ontario", "store_format": "Downtown", "opening_year": 2018},
    {"location_id": "LOC002", "location_name": "North York",       "city": "Toronto",     "region": "Ontario", "store_format": "Urban",    "opening_year": 2019},
    {"location_id": "LOC003", "location_name": "Scarborough",      "city": "Toronto",     "region": "Ontario", "store_format": "Urban",    "opening_year": 2020},
    {"location_id": "LOC004", "location_name": "Mississauga",      "city": "Mississauga", "region": "Ontario", "store_format": "Suburban", "opening_year": 2017},
    {"location_id": "LOC005", "location_name": "Vaughan",          "city": "Vaughan",     "region": "Ontario", "store_format": "Suburban", "opening_year": 2021},
    {"location_id": "LOC006", "location_name": "Hamilton",         "city": "Hamilton",    "region": "Ontario", "store_format": "Suburban", "opening_year": 2022},
    {"location_id": "LOC007", "location_name": "Ottawa",           "city": "Ottawa",      "region": "Ontario", "store_format": "Urban",    "opening_year": 2018},
    {"location_id": "LOC008", "location_name": "London",           "city": "London",      "region": "Ontario", "store_format": "Suburban", "opening_year": 2023},
]
locations = pd.DataFrame(locations_data)

products_data = [
    # Proteins
    {"product_id": "P001", "product_name": "Grilled Chicken Breast", "category": "Proteins",   "subcategory": "Chicken",    "menu_price": 14.99, "standard_food_cost": 4.80, "portion_size_grams": 180, "vendor_id": "V003"},
    {"product_id": "P002", "product_name": "Beef Striploin",         "category": "Proteins",   "subcategory": "Beef",       "menu_price": 18.99, "standard_food_cost": 6.90, "portion_size_grams": 200, "vendor_id": "V003"},
    {"product_id": "P003", "product_name": "Salmon Fillet",          "category": "Proteins",   "subcategory": "Fish",       "menu_price": 19.49, "standard_food_cost": 7.10, "portion_size_grams": 170, "vendor_id": "V003"},
    {"product_id": "P004", "product_name": "Turkey Slices",          "category": "Proteins",   "subcategory": "Turkey",     "menu_price": 13.99, "standard_food_cost": 4.20, "portion_size_grams": 160, "vendor_id": "V003"},
    {"product_id": "P005", "product_name": "Tofu Cubes",             "category": "Proteins",   "subcategory": "Plant",      "menu_price": 11.99, "standard_food_cost": 3.10, "portion_size_grams": 150, "vendor_id": "V001"},

    # Produce
    {"product_id": "P006", "product_name": "Mixed Greens",           "category": "Produce",    "subcategory": "Leafy",      "menu_price": 7.99,  "standard_food_cost": 2.10, "portion_size_grams": 120, "vendor_id": "V004"},
    {"product_id": "P007", "product_name": "Cherry Tomatoes",        "category": "Produce",    "subcategory": "Tomatoes",   "menu_price": 6.99,  "standard_food_cost": 1.80, "portion_size_grams": 100, "vendor_id": "V004"},
    {"product_id": "P008", "product_name": "Avocado Slices",         "category": "Produce",    "subcategory": "Avocado",    "menu_price": 8.99,  "standard_food_cost": 2.90, "portion_size_grams": 90,  "vendor_id": "V004"},
    {"product_id": "P009", "product_name": "Roasted Vegetables",     "category": "Produce",    "subcategory": "Hot Veg",    "menu_price": 7.49,  "standard_food_cost": 2.20, "portion_size_grams": 140, "vendor_id": "V004"},
    {"product_id": "P010", "product_name": "Fruit Cup",              "category": "Produce",    "subcategory": "Fruit",      "menu_price": 5.99,  "standard_food_cost": 1.95, "portion_size_grams": 130, "vendor_id": "V004"},

    # Beverages
    {"product_id": "P011", "product_name": "Bottled Water",          "category": "Beverages",  "subcategory": "Cold",       "menu_price": 2.99,  "standard_food_cost": 0.65, "portion_size_grams": 500, "vendor_id": "V002"},
    {"product_id": "P012", "product_name": "Sparkling Water",        "category": "Beverages",  "subcategory": "Cold",       "menu_price": 3.49,  "standard_food_cost": 0.85, "portion_size_grams": 500, "vendor_id": "V002"},
    {"product_id": "P013", "product_name": "Iced Tea",               "category": "Beverages",  "subcategory": "Cold",       "menu_price": 3.99,  "standard_food_cost": 0.95, "portion_size_grams": 450, "vendor_id": "V002"},
    {"product_id": "P014", "product_name": "Coffee",                 "category": "Beverages",  "subcategory": "Hot",        "menu_price": 2.79,  "standard_food_cost": 0.55, "portion_size_grams": 350, "vendor_id": "V002"},
    {"product_id": "P015", "product_name": "Fresh Juice",            "category": "Beverages",  "subcategory": "Cold",       "menu_price": 4.99,  "standard_food_cost": 1.60, "portion_size_grams": 400, "vendor_id": "V004"},

    # Sides
    {"product_id": "P016", "product_name": "French Fries",           "category": "Sides",      "subcategory": "Potato",     "menu_price": 4.99,  "standard_food_cost": 1.30, "portion_size_grams": 150, "vendor_id": "V001"},
    {"product_id": "P017", "product_name": "Rice Pilaf",             "category": "Sides",      "subcategory": "Rice",       "menu_price": 4.49,  "standard_food_cost": 1.10, "portion_size_grams": 160, "vendor_id": "V001"},
    {"product_id": "P018", "product_name": "Mashed Potatoes",        "category": "Sides",      "subcategory": "Potato",     "menu_price": 4.79,  "standard_food_cost": 1.25, "portion_size_grams": 170, "vendor_id": "V001"},
    {"product_id": "P019", "product_name": "Mac and Cheese",         "category": "Sides",      "subcategory": "Pasta",      "menu_price": 5.99,  "standard_food_cost": 1.85, "portion_size_grams": 180, "vendor_id": "V001"},
    {"product_id": "P020", "product_name": "Dinner Roll",            "category": "Sides",      "subcategory": "Bakery",     "menu_price": 2.49,  "standard_food_cost": 0.60, "portion_size_grams": 70,  "vendor_id": "V001"},

    # Desserts
    {"product_id": "P021", "product_name": "Chocolate Brownie",      "category": "Desserts",   "subcategory": "Baked",      "menu_price": 4.99,  "standard_food_cost": 1.15, "portion_size_grams": 95,  "vendor_id": "V001"},
    {"product_id": "P022", "product_name": "Cheesecake Slice",       "category": "Desserts",   "subcategory": "Cake",       "menu_price": 5.99,  "standard_food_cost": 1.65, "portion_size_grams": 110, "vendor_id": "V001"},
    {"product_id": "P023", "product_name": "Fruit Parfait",          "category": "Desserts",   "subcategory": "Cold",       "menu_price": 5.49,  "standard_food_cost": 1.40, "portion_size_grams": 120, "vendor_id": "V004"},
    {"product_id": "P024", "product_name": "Cookie Pack",            "category": "Desserts",   "subcategory": "Baked",      "menu_price": 3.99,  "standard_food_cost": 0.95, "portion_size_grams": 80,  "vendor_id": "V001"},

    # Disposables
    {"product_id": "P025", "product_name": "Takeout Container",      "category": "Disposables","subcategory": "Packaging",  "menu_price": 0.50,  "standard_food_cost": 0.18, "portion_size_grams": 1,   "vendor_id": "V005"},
    {"product_id": "P026", "product_name": "Drink Lid",              "category": "Disposables","subcategory": "Packaging",  "menu_price": 0.20,  "standard_food_cost": 0.05, "portion_size_grams": 1,   "vendor_id": "V005"},
    {"product_id": "P027", "product_name": "Napkin Pack",            "category": "Disposables","subcategory": "Supplies",   "menu_price": 0.15,  "standard_food_cost": 0.03, "portion_size_grams": 1,   "vendor_id": "V005"},
    {"product_id": "P028", "product_name": "Cutlery Set",            "category": "Disposables","subcategory": "Supplies",   "menu_price": 0.35,  "standard_food_cost": 0.09, "portion_size_grams": 1,   "vendor_id": "V005"},
]
products = pd.DataFrame(products_data)

vendor_lookup = pd.DataFrame(vendors)

# -----------------------------
# 6. CREATE DATA DICTIONARY
# -----------------------------
data_dictionary_rows = []

def add_dict_rows(table_name, rows):
    for row in rows:
        data_dictionary_rows.append({
            "table_name": table_name,
            "column_name": row["column_name"],
            "data_type": row["data_type"],
            "grain": row["grain"],
            "description": row["description"],
            "example_value": row["example_value"]
        })

add_dict_rows("locations", [
    {"column_name": "location_id", "data_type": "string", "grain": "location", "description": "Unique location identifier", "example_value": "LOC001"},
    {"column_name": "location_name", "data_type": "string", "grain": "location", "description": "Store name", "example_value": "Toronto Downtown"},
    {"column_name": "city", "data_type": "string", "grain": "location", "description": "Store city", "example_value": "Toronto"},
    {"column_name": "region", "data_type": "string", "grain": "location", "description": "Store region", "example_value": "Ontario"},
    {"column_name": "store_format", "data_type": "string", "grain": "location", "description": "Store format classification", "example_value": "Downtown"},
    {"column_name": "opening_year", "data_type": "integer", "grain": "location", "description": "Store opening year", "example_value": "2018"},
])

add_dict_rows("products", [
    {"column_name": "product_id", "data_type": "string", "grain": "product", "description": "Unique product identifier", "example_value": "P001"},
    {"column_name": "product_name", "data_type": "string", "grain": "product", "description": "Product name", "example_value": "Grilled Chicken Breast"},
    {"column_name": "category", "data_type": "string", "grain": "product", "description": "Product category", "example_value": "Proteins"},
    {"column_name": "subcategory", "data_type": "string", "grain": "product", "description": "Product subcategory", "example_value": "Chicken"},
    {"column_name": "menu_price", "data_type": "decimal", "grain": "product", "description": "Standard menu selling price", "example_value": "14.99"},
    {"column_name": "standard_food_cost", "data_type": "decimal", "grain": "product", "description": "Standard product food cost", "example_value": "4.80"},
    {"column_name": "portion_size_grams", "data_type": "integer", "grain": "product", "description": "Typical portion size in grams", "example_value": "180"},
    {"column_name": "vendor_id", "data_type": "string", "grain": "product", "description": "Assigned primary vendor", "example_value": "V003"},
])

add_dict_rows("vendor_prices", [
    {"column_name": "month", "data_type": "string", "grain": "vendor-product-month", "description": "Month of vendor cost snapshot", "example_value": "2025-03"},
    {"column_name": "vendor_id", "data_type": "string", "grain": "vendor-product-month", "description": "Vendor identifier", "example_value": "V003"},
    {"column_name": "product_id", "data_type": "string", "grain": "vendor-product-month", "description": "Product identifier", "example_value": "P001"},
    {"column_name": "vendor_name", "data_type": "string", "grain": "vendor-product-month", "description": "Vendor name", "example_value": "Prime Protein Partners"},
    {"column_name": "actual_unit_cost", "data_type": "decimal", "grain": "vendor-product-month", "description": "Vendor actual unit cost for the month", "example_value": "5.26"},
    {"column_name": "prior_unit_cost", "data_type": "decimal", "grain": "vendor-product-month", "description": "Previous month's unit cost", "example_value": "5.09"},
    {"column_name": "cost_change_pct", "data_type": "decimal", "grain": "vendor-product-month", "description": "Percent change versus previous month", "example_value": "3.34"},
])

add_dict_rows("sales", [
    {"column_name": "transaction_date", "data_type": "date", "grain": "product-location-day", "description": "Daily transaction date", "example_value": "2025-03-14"},
    {"column_name": "month", "data_type": "string", "grain": "product-location-day", "description": "Month bucket", "example_value": "2025-03"},
    {"column_name": "location_id", "data_type": "string", "grain": "product-location-day", "description": "Location identifier", "example_value": "LOC001"},
    {"column_name": "product_id", "data_type": "string", "grain": "product-location-day", "description": "Product identifier", "example_value": "P001"},
    {"column_name": "units_sold", "data_type": "integer", "grain": "product-location-day", "description": "Units sold", "example_value": "28"},
    {"column_name": "revenue", "data_type": "decimal", "grain": "product-location-day", "description": "Net sales revenue after discount", "example_value": "403.20"},
    {"column_name": "discount_amount", "data_type": "decimal", "grain": "product-location-day", "description": "Discount value applied", "example_value": "12.60"},
    {"column_name": "avg_ticket", "data_type": "decimal", "grain": "product-location-day", "description": "Average ticket estimate for the row context", "example_value": "18.90"},
    {"column_name": "transactions_count", "data_type": "integer", "grain": "product-location-day", "description": "Estimated transaction count", "example_value": "21"},
])

add_dict_rows("costs", [
    {"column_name": "month", "data_type": "string", "grain": "product-location-month", "description": "Month bucket", "example_value": "2025-03"},
    {"column_name": "location_id", "data_type": "string", "grain": "product-location-month", "description": "Location identifier", "example_value": "LOC001"},
    {"column_name": "product_id", "data_type": "string", "grain": "product-location-month", "description": "Product identifier", "example_value": "P001"},
    {"column_name": "cogs", "data_type": "decimal", "grain": "product-location-month", "description": "Cost of goods sold", "example_value": "4,185.90"},
    {"column_name": "labor_cost", "data_type": "decimal", "grain": "product-location-month", "description": "Allocated labor cost", "example_value": "1,502.44"},
    {"column_name": "packaging_cost", "data_type": "decimal", "grain": "product-location-month", "description": "Packaging and serving cost", "example_value": "245.10"},
    {"column_name": "overhead_allocated", "data_type": "decimal", "grain": "product-location-month", "description": "Allocated overhead", "example_value": "918.56"},
    {"column_name": "total_cost", "data_type": "decimal", "grain": "product-location-month", "description": "Total operating cost", "example_value": "6,851.99"},
])

add_dict_rows("waste", [
    {"column_name": "month", "data_type": "string", "grain": "product-location-month", "description": "Month bucket", "example_value": "2025-03"},
    {"column_name": "location_id", "data_type": "string", "grain": "product-location-month", "description": "Location identifier", "example_value": "LOC006"},
    {"column_name": "product_id", "data_type": "string", "grain": "product-location-month", "description": "Product identifier", "example_value": "P001"},
    {"column_name": "waste_units", "data_type": "integer", "grain": "product-location-month", "description": "Wasted units for the month", "example_value": "34"},
    {"column_name": "waste_cost", "data_type": "decimal", "grain": "product-location-month", "description": "Estimated waste cost", "example_value": "172.40"},
    {"column_name": "waste_reason", "data_type": "string", "grain": "product-location-month", "description": "Main waste reason", "example_value": "Spoilage"},
    {"column_name": "recorded_by", "data_type": "string", "grain": "product-location-month", "description": "Recorded by role", "example_value": "Kitchen Lead"},
])

data_dictionary = pd.DataFrame(data_dictionary_rows)

# -----------------------------
# 7. CREATE VENDOR PRICES
# -----------------------------
vendor_monthly_inflation = {
    "V001": 0.0075,
    "V002": 0.0090,
    "V003": 0.0260,  # problem vendor
    "V004": 0.0150,
    "V005": 0.0110,
}

vendor_prices_rows = []

for _, product in products.iterrows():
    product_id = product["product_id"]
    category = product["category"]
    vendor_id = product["vendor_id"]
    vendor_name = vendor_lookup.loc[vendor_lookup["vendor_id"] == vendor_id, "vendor_name"].iloc[0]
    prior_unit_cost = float(product["standard_food_cost"])

    for i, month in enumerate(months):
        base_inflation = vendor_monthly_inflation[vendor_id]

        if vendor_id == inflation_vendor_id and category == "Proteins":
            category_adjustment = 0.010
        elif category == "Produce":
            category_adjustment = 0.004
        else:
            category_adjustment = 0.000

        if month >= "2025-07":
            late_period_pressure = 0.003
        else:
            late_period_pressure = 0.000

        noise = np.random.uniform(-0.0015, 0.0015)

        if i == 0:
            actual_unit_cost = round(prior_unit_cost, 2)
            cost_change_pct = 0.00
        else:
            growth_rate = max(base_inflation + category_adjustment + late_period_pressure + noise, 0.0)
            actual_unit_cost = round(prior_unit_cost * (1 + growth_rate), 2)
            cost_change_pct = round(((actual_unit_cost - prior_unit_cost) / prior_unit_cost) * 100, 2)

        vendor_prices_rows.append({
            "month": month,
            "vendor_id": vendor_id,
            "product_id": product_id,
            "vendor_name": vendor_name,
            "actual_unit_cost": actual_unit_cost,
            "prior_unit_cost": round(prior_unit_cost, 2),
            "cost_change_pct": cost_change_pct
        })

        prior_unit_cost = actual_unit_cost

vendor_prices = pd.DataFrame(vendor_prices_rows)

# -----------------------------
# 8. CREATE DAILY SALES TABLE
# -----------------------------
# Build a lookup for latest unit costs by month/product
vendor_cost_lookup = vendor_prices.set_index(["month", "product_id"])["actual_unit_cost"].to_dict()

sales_rows = []

# Product demand multipliers to create realistic variation
product_demand_multiplier = {
    "P001": 1.00, "P002": 0.82, "P003": 0.74, "P004": 0.86, "P005": 0.68,
    "P006": 0.96, "P007": 0.88, "P008": 0.72, "P009": 0.82, "P010": 0.70,
    "P011": 1.25, "P012": 1.10, "P013": 1.18, "P014": 1.35, "P015": 0.92,
    "P016": 1.22, "P017": 1.02, "P018": 0.98, "P019": 0.84, "P020": 1.15,
    "P021": 0.70, "P022": 0.62, "P023": 0.58, "P024": 0.76,
    "P025": 1.18, "P026": 1.22, "P027": 1.12, "P028": 1.10,
}

# Promotions/seasonality
summer_beverage_boost_months = {"2024-06", "2024-07", "2024-08", "2025-06", "2025-07", "2025-08", "2026-04"}
holiday_dessert_boost_months = {"2024-11", "2024-12", "2025-11", "2025-12"}

for month in months:
    year = int(month.split("-")[0])
    month_num = int(month.split("-")[1])
    days_in_month = calendar.monthrange(year, month_num)[1]
    month_dates = pd.date_range(start=f"{month}-01", end=f"{month}-{days_in_month:02d}", freq="D")

    for current_date in month_dates:
        day_of_week = current_date.weekday()
        dow_mult = dow_factor[day_of_week]

        for _, loc in locations.iterrows():
            location_id = loc["location_id"]
            loc_sales_mult = location_sales_factor[location_id]
            loc_avg_ticket_base = avg_ticket_base_by_location[location_id]

            for _, prod in products.iterrows():
                product_id = prod["product_id"]
                category = prod["category"]
                menu_price = float(prod["menu_price"])
                demand_seed = base_units_by_category[category]
                prod_mult = product_demand_multiplier[product_id]

                sales_growth_mult = monthly_sales_growth[month]

                seasonality_mult = 1.00
                if category == "Beverages" and month in summer_beverage_boost_months:
                    seasonality_mult += 0.10
                if category == "Desserts" and month in holiday_dessert_boost_months:
                    seasonality_mult += 0.12

                underperformance_penalty = 1.00
                if location_id in underperforming_locations:
                    if month >= "2025-01":
                        underperformance_penalty -= 0.08
                    if month >= "2025-07":
                        underperformance_penalty -= 0.04
                    if category in problem_categories:
                        underperformance_penalty -= 0.03

                # High-volume low-margin categories
                hv_mult = 1.08 if category in high_volume_low_margin_categories else 1.00

                # Small noise
                noise = np.random.uniform(0.92, 1.08)

                expected_units = (
                    demand_seed
                    * prod_mult
                    * loc_sales_mult
                    * sales_growth_mult
                    * dow_mult
                    * seasonality_mult
                    * underperformance_penalty
                    * hv_mult
                    * noise
                )

                units_sold = max(int(round(expected_units)), 0)

                # Disposables track volume of food/beverage traffic
                if category == "Disposables":
                    units_sold = max(int(round(expected_units * 0.75)), 0)

                gross_revenue = units_sold * menu_price
                discount_rate = discount_rate_by_category[category]

                # Slightly higher discounting in underperformers
                if location_id in underperforming_locations and month >= "2025-01":
                    discount_rate += 0.01

                # Small promo boost on selected beverages and desserts
                if category in ["Beverages", "Desserts"] and month in holiday_dessert_boost_months.union(summer_beverage_boost_months):
                    discount_rate += 0.005

                discount_amount = gross_revenue * discount_rate
                revenue = gross_revenue - discount_amount

                avg_ticket = loc_avg_ticket_base * np.random.uniform(0.97, 1.03)
                if location_id in underperforming_locations:
                    avg_ticket *= 0.97

                transactions_count = max(int(round(revenue / max(avg_ticket, 1))), 1) if revenue > 0 else 0

                sales_rows.append({
                    "transaction_date": current_date.strftime("%Y-%m-%d"),
                    "month": month,
                    "location_id": location_id,
                    "product_id": product_id,
                    "units_sold": units_sold,
                    "revenue": round(revenue, 2),
                    "discount_amount": round(discount_amount, 2),
                    "avg_ticket": round(avg_ticket, 2),
                    "transactions_count": transactions_count
                })

sales = pd.DataFrame(sales_rows)

# -----------------------------
# 9. CREATE MONTHLY WASTE TABLE
# -----------------------------
monthly_units = (
    sales.groupby(["month", "location_id", "product_id"], as_index=False)["units_sold"]
    .sum()
    .rename(columns={"units_sold": "monthly_units_sold"})
)

waste_rows = []

reason_names = list(waste_reason_weights.keys())
reason_probs = list(waste_reason_weights.values())

for _, row in monthly_units.iterrows():
    month = row["month"]
    location_id = row["location_id"]
    product_id = row["product_id"]
    monthly_units_sold = int(row["monthly_units_sold"])

    prod = products.loc[products["product_id"] == product_id].iloc[0]
    category = prod["category"]
    standard_cost = float(prod["standard_food_cost"])

    category_mult = waste_multiplier_by_category[category]
    location_mult = location_waste_factor[location_id]

    # later periods worsen
    late_period_mult = 1.10 if month >= "2025-07" else 1.00

    # proteins + produce have more waste
    base_waste_rate = 0.020
    waste_rate = base_waste_rate * category_mult * location_mult * late_period_mult * np.random.uniform(0.90, 1.10)

    # underperforming stores waste more
    if location_id in underperforming_locations:
        waste_rate *= 1.10

    waste_units = int(round(monthly_units_sold * waste_rate))

    actual_unit_cost = vendor_cost_lookup[(month, product_id)]
    waste_cost = waste_units * actual_unit_cost

    waste_reason = np.random.choice(reason_names, p=reason_probs)
    recorded_by = np.random.choice(recorded_by_options)

    waste_rows.append({
        "month": month,
        "location_id": location_id,
        "product_id": product_id,
        "waste_units": waste_units,
        "waste_cost": round(waste_cost, 2),
        "waste_reason": waste_reason,
        "recorded_by": recorded_by
    })

waste = pd.DataFrame(waste_rows)

# -----------------------------
# 10. CREATE MONTHLY COSTS TABLE
# -----------------------------
monthly_sales = (
    sales.groupby(["month", "location_id", "product_id"], as_index=False)
    .agg({
        "units_sold": "sum",
        "revenue": "sum",
        "transactions_count": "sum"
    })
)

waste_lookup = waste.set_index(["month", "location_id", "product_id"])[["waste_units", "waste_cost"]].to_dict("index")

cost_rows = []

for _, row in monthly_sales.iterrows():
    month = row["month"]
    location_id = row["location_id"]
    product_id = row["product_id"]
    units_sold = int(row["units_sold"])
    revenue = float(row["revenue"])
    transactions_count = int(row["transactions_count"])

    prod = products.loc[products["product_id"] == product_id].iloc[0]
    category = prod["category"]

    actual_unit_cost = vendor_cost_lookup[(month, product_id)]
    waste_info = waste_lookup[(month, location_id, product_id)]
    waste_units = int(waste_info["waste_units"])
    waste_cost = float(waste_info["waste_cost"])

    cogs_growth_mult = monthly_cogs_growth[month] / monthly_sales_growth[month]

    base_cogs = (units_sold + waste_units) * actual_unit_cost * cogs_growth_mult

    # Higher COGS pressure for problem categories
    if category in problem_categories:
        base_cogs *= 1.03

    # labor logic
    labor_pct = {
        "Proteins": 0.16,
        "Produce": 0.13,
        "Beverages": 0.08,
        "Sides": 0.10,
        "Desserts": 0.09,
        "Disposables": 0.02
    }[category]

    labor_cost = revenue * labor_pct * location_labor_factor[location_id]

    # packaging logic
    packaging_pct = {
        "Proteins": 0.022,
        "Produce": 0.020,
        "Beverages": 0.018,
        "Sides": 0.019,
        "Desserts": 0.016,
        "Disposables": 0.010
    }[category]

    packaging_cost = revenue * packaging_pct

    # overhead allocation
    overhead_base_pct = 0.11
    overhead_allocated = revenue * overhead_base_pct

    # underperformers suffer slightly worse overhead efficiency
    if location_id in underperforming_locations:
        overhead_allocated *= 1.06

    total_cost = base_cogs + labor_cost + packaging_cost + overhead_allocated

    cost_rows.append({
        "month": month,
        "location_id": location_id,
        "product_id": product_id,
        "cogs": round(base_cogs, 2),
        "labor_cost": round(labor_cost, 2),
        "packaging_cost": round(packaging_cost, 2),
        "overhead_allocated": round(overhead_allocated, 2),
        "total_cost": round(total_cost, 2)
    })

costs = pd.DataFrame(cost_rows)

# -----------------------------
# 11. QUALITY CHECKS
# -----------------------------
print("\n" + "=" * 100)
print("QUALITY CHECKS")
print("=" * 100)

assert locations["location_id"].is_unique, "Duplicate location_id found"
assert products["product_id"].is_unique, "Duplicate product_id found"
assert sales["revenue"].ge(0).all(), "Negative revenue found"
assert sales["units_sold"].ge(0).all(), "Negative units found"
assert costs["total_cost"].ge(0).all(), "Negative total cost found"
assert waste["waste_units"].ge(0).all(), "Negative waste units found"
assert set(sales["location_id"]).issubset(set(locations["location_id"])), "Unknown location in sales"
assert set(sales["product_id"]).issubset(set(products["product_id"])), "Unknown product in sales"
assert set(costs["location_id"]).issubset(set(locations["location_id"])), "Unknown location in costs"
assert set(costs["product_id"]).issubset(set(products["product_id"])), "Unknown product in costs"
assert set(waste["location_id"]).issubset(set(locations["location_id"])), "Unknown location in waste"
assert set(waste["product_id"]).issubset(set(products["product_id"])), "Unknown product in waste"

print("All core quality checks passed.")

# -----------------------------
# 12. CREATE KPI CHECK TABLE
# -----------------------------
monthly_revenue_total = sales.groupby("month", as_index=False)["revenue"].sum()
monthly_cost_total = costs.groupby("month", as_index=False)["total_cost"].sum()
kpi_check = monthly_revenue_total.merge(monthly_cost_total, on="month", how="left")
kpi_check["gross_profit"] = kpi_check["revenue"] - kpi_check["total_cost"]
kpi_check["gross_margin_pct"] = np.where(
    kpi_check["revenue"] > 0,
    (kpi_check["gross_profit"] / kpi_check["revenue"]) * 100,
    0
)

print("\n" + "=" * 100)
print("KPI CHECK: MONTHLY GROSS MARGIN %")
print("=" * 100)
display(kpi_check.tail(12))

# -----------------------------
# 13. EXPORT FILES
# -----------------------------
locations.to_csv(RAW_DIR / "locations.csv", index=False)
products.to_csv(RAW_DIR / "products.csv", index=False)
vendor_prices.to_csv(RAW_DIR / "vendor_prices.csv", index=False)
sales.to_csv(RAW_DIR / "sales.csv", index=False)
costs.to_csv(RAW_DIR / "costs.csv", index=False)
waste.to_csv(RAW_DIR / "waste.csv", index=False)
data_dictionary.to_csv(DATA_DIR / "data_dictionary.csv", index=False)
kpi_check.to_csv(CLEANED_DIR / "kpi_check_monthly.csv", index=False)

# -----------------------------
# 14. PREVIEWS
# -----------------------------
print("\n" + "=" * 100)
print("TABLE SHAPES")
print("=" * 100)
print("locations      :", locations.shape)
print("products       :", products.shape)
print("vendor_prices  :", vendor_prices.shape)
print("sales          :", sales.shape)
print("costs          :", costs.shape)
print("waste          :", waste.shape)
print("data_dictionary:", data_dictionary.shape)

print("\n" + "=" * 100)
print("LOCATIONS PREVIEW")
print("=" * 100)
display(locations)

print("\n" + "=" * 100)
print("PRODUCTS PREVIEW")
print("=" * 100)
display(products.head(10))

print("\n" + "=" * 100)
print("VENDOR PRICES PREVIEW")
print("=" * 100)
display(vendor_prices.head(12))

print("\n" + "=" * 100)
print("SALES PREVIEW")
print("=" * 100)
display(sales.head(12))

print("\n" + "=" * 100)
print("COSTS PREVIEW")
print("=" * 100)
display(costs.head(12))

print("\n" + "=" * 100)
print("WASTE PREVIEW")
print("=" * 100)
display(waste.head(12))

print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)
print(RAW_DIR / "locations.csv")
print(RAW_DIR / "products.csv")
print(RAW_DIR / "vendor_prices.csv")
print(RAW_DIR / "sales.csv")
print(RAW_DIR / "costs.csv")
print(RAW_DIR / "waste.csv")
print(DATA_DIR / "data_dictionary.csv")
print(CLEANED_DIR / "kpi_check_monthly.csv")

Mounted at /content/drive
PROJECT FOLDERS READY
/content/drive/MyDrive/operations-profitability-cost-optimization

BUSINESS ASSUMPTIONS
Synthetic company: Harvest Table Group
North star metric: Gross Margin %
Time period: 2024-01 to 2026-04
Problem categories: ['Proteins', 'Produce']
Underperforming locations: ['LOC006', 'LOC008']
Inflation vendor: V003

QUALITY CHECKS
All core quality checks passed.

KPI CHECK: MONTHLY GROSS MARGIN %


,month,revenue,total_cost,gross_profit,gross_margin_pct
16,2025-05,"1,039,795.26","774,502.45","265,292.81",25.51
17,2025-06,"1,033,516.98","781,266.80","252,250.18",24.41
18,2025-07,"1,075,303.63","834,733.61","240,570.02",22.37
19,2025-08,"1,104,357.68","878,784.87","225,572.81",20.43
20,2025-09,"1,044,072.99","856,947.20","187,125.79",17.92
21,2025-10,"1,101,590.69","927,247.31","174,343.38",15.83
22,2025-11,"1,090,478.08","941,217.44","149,260.64",13.69
23,2025-12,"1,132,278.47","1,003,359.26","128,919.21",11.39
24,2026-01,"1,147,993.37","1,049,644.90","98,348.47",8.57
25,2026-02,"1,046,363.82","982,508.97","63,854.85",6.10



TABLE SHAPES
locations      : (8, 6)
products       : (28, 8)
vendor_prices  : (784, 7)
sales          : (190624, 9)
costs          : (6272, 8)
waste          : (6272, 7)
data_dictionary: (45, 6)

LOCATIONS PREVIEW


,location_id,location_name,city,region,store_format,opening_year
0,LOC001,Toronto Downtown,Toronto,Ontario,Downtown,2018
1,LOC002,North York,Toronto,Ontario,Urban,2019
2,LOC003,Scarborough,Toronto,Ontario,Urban,2020
3,LOC004,Mississauga,Mississauga,Ontario,Suburban,2017
4,LOC005,Vaughan,Vaughan,Ontario,Suburban,2021
5,LOC006,Hamilton,Hamilton,Ontario,Suburban,2022
6,LOC007,Ottawa,Ottawa,Ontario,Urban,2018
7,LOC008,London,London,Ontario,Suburban,2023



PRODUCTS PREVIEW


...


COSTS PREVIEW


,month,location_id,product_id,cogs,labor_cost,packaging_cost,overhead_allocated,total_cost
0,2024-01,LOC001,P001,"3,544.85","1,730.72",220.35,"1,101.73","6,597.65"
1,2024-01,LOC001,P002,"4,207.34","1,814.54",231.02,"1,155.09","7,407.98"
2,2024-01,LOC001,P003,"3,810.07","1,636.00",208.29,"1,041.43","6,695.79"
3,2024-01,LOC001,P004,"2,612.90","1,355.34",172.56,862.78,"5,003.58"
4,2024-01,LOC001,P005,"1,545.41",932.86,118.77,593.84,"3,190.87"
5,2024-01,LOC001,P006,"1,630.90",798.71,113.78,625.77,"3,169.16"
6,2024-01,LOC001,P007,"1,268.14",633.07,90.18,495.99,"2,487.37"
7,2024-01,LOC001,P008,"1,684.67",672.17,95.75,526.63,"2,979.22"
8,2024-01,LOC001,P009,"1,463.84",641.62,91.40,502.69,"2,699.54"
9,2024-01,LOC001,P010,"1,116.73",442.16,62.99,346.42,"1,968.29"



WASTE PREVIEW


,month,location_id,product_id,waste_units,waste_cost,waste_reason,recorded_by
0,2024-01,LOC001,P001,21,100.80,Overproduction,Kitchen Lead
1,2024-01,LOC001,P002,16,110.40,Overproduction,Shift Manager
2,2024-01,LOC001,P003,15,106.50,Overproduction,Inventory Coordinator
3,2024-01,LOC001,P004,20,84.00,Expired Inventory,Shift Manager
4,2024-01,LOC001,P005,15,46.50,Overproduction,Kitchen Lead
5,2024-01,LOC001,P006,20,42.00,Overproduction,Inventory Coordinator
6,2024-01,LOC001,P007,19,34.20,Prep Loss,Operations Supervisor
7,2024-01,LOC001,P008,15,43.50,Spoilage,Operations Supervisor
8,2024-01,LOC001,P009,17,37.40,Damaged Handling,Operations Supervisor
9,2024-01,LOC001,P010,14,27.30,Prep Loss,Inventory Coordinator



FILES SAVED
/content/drive/MyDrive/operations-profitability-cost-optimization/data/raw/locations.csv
/content/drive/MyDrive/operations-profitability-cost-optimization/data/raw/products.csv
/content/drive/MyDrive/operations-profitability-cost-optimization/data/raw/vendor_prices.csv
/content/drive/MyDrive/operations-profitability-cost-optimization/data/raw/sales.csv
/content/drive/MyDrive/operations-profitability-cost-optimization/data/raw/costs.csv
/content/drive/MyDrive/operations-profitability-cost-optimization/data/raw/waste.csv
/content/drive/MyDrive/operations-profitability-cost-optimization/data/data_dictionary.csv
/content/drive/MyDrive/operations-profitability-cost-optimization/data/cleaned/kpi_check_monthly.csv
